In [2]:
pip install biopython -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 50.1 MB/s eta 0:00:00


In [3]:
pip install torch_geometric -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 34.0 MB/s eta 0:00:00


In [4]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/My Drive/Research/2026/GNNToyProblem/')

Mounted at /content/drive


# Enzyme Function Prediction Using Graph Neural Networks and Protein Structure
This notebook is a toy project for learning Graph Neural Networks and applying them to protein structure. More precisely, given the structure of an enzyme this model will predict if it is an oxyreductase or a transferase.

##1. Data Manipulation

The protein structures were downloaded from RCSB PDB in format mmCIF. We selected proteins from Escherichia coli with structure experimentally determined by X-ray diffraction with a refinement resolution of at most 2.5Å having Enzyme Comission number exclusively 1 or exclusively 2.

###1.1 Obtaining $\alpha$-carbons scaffold of the structure
The $C_{\alpha}$ of an amino acid can be viewed as its "central" point, keeping only these carbons in the structure of the protein can be seen as a simplification that retains the essential information about the molecule structure.

In [ ]:
from Bio.PDB import MMCIFParser

#This function return the list of the spatial coordinates of the alpha-carbons
#of a given mmCIF file. It uses a mmCIF parser from Biopython.
def get_alpha_carbons(cif_file):
  parser = MMCIFParser(QUIET=True)
  structure = parser.get_structure("protein", cif_file)

  ca_coords = []

  for model in structure:
    for chain in model:
      for residue in chain:
        if "CA" in residue:
          atom = residue["CA"]
          coord = atom.get_coord()
          aa = residue.resname
          ca_coords.append((aa,coord))

  return ca_coords

In [ ]:
import os

#(Truncated) output from the above defined function
get_alpha_carbons('data/EC_01/1aa6.cif')[:5]

[('MET', array([68.592,  3.44 , 30.52 ], dtype=float32)),
 ('LYS', array([71.858,  4.86 , 31.889], dtype=float32)),
 ('LYS', array([73.133,  8.159, 33.285], dtype=float32)),
 ('VAL', array([76.264,  9.713, 31.806], dtype=float32)),
 ('VAL', array([77.941, 12.384, 33.981], dtype=float32))]

###1.2 Creating graph representation

We now create a graph that represents the protein structure, the nodes will be $C_{\alpha}$ and there will be an edge betwen two carbons when their distance is less than a defined distance cutoff. The graph will be represented using edge index.

In [ ]:
import numpy as np

def generate_graph(ca_list, dist_cutoff=8):

  edge_index_start = []
  edge_index_end = []

  N = len(ca_list)

  for i in range(N):
    for j in range(i,N):
      if np.linalg.norm(ca_list[i][1]-ca_list[j][1]) < dist_cutoff:
        edge_index_start.append(i)
        edge_index_end.append(j)

        if i != j:
          edge_index_start.append(j)
          edge_index_end.append(i)

  edge_index = np.array([edge_index_start, edge_index_end])
  return edge_index

In [ ]:
#(Truncated) example of an edge list
ca_list = get_alpha_carbons('data/EC_01/1aa6.cif')
a, b = generate_graph(ca_list)

for i,j in zip(a,b):
  print(i,j)

  if i > 5:
    break

0 0
0 1
1 0
0 2
2 0
0 19
19 0


In [ ]:
from tqdm import tqdm

data_dirs = ['data/EC_01', 'data/EC_02']
all_alpha_carbons_01 = []
all_alpha_carbons_02 = []

for data_dir in data_dirs:
  print(f"Processing files in {data_dir}")
  for filename in tqdm(os.listdir(data_dir)):
    file_path = os.path.join(data_dir, filename)
    ca_coords = get_alpha_carbons(file_path)
    pdb_id = file_path[-8:-4]
    if data_dir == 'data/EC_01':
      all_alpha_carbons_01.append((pdb_id, ca_coords))
    elif data_dir == 'data/EC_02':
      all_alpha_carbons_02.append((pdb_id, ca_coords))

Processing files in data/EC_01


100%|██████████| 471/471 [03:45<00:00,  2.09it/s]


Processing files in data/EC_02


100%|██████████| 752/752 [05:18<00:00,  2.36it/s]


In [ ]:
import pickle

output_dir = 'data_processed'
os.makedirs(output_dir, exist_ok=True)

output_path_01 = os.path.join(output_dir, 'all_alpha_carbons_01.pkl')
with open(output_path_01, 'wb') as f:
    pickle.dump(all_alpha_carbons_01, f)
print(f"Saved all_alpha_carbons_01 to {output_path_01}")

output_path_02 = os.path.join(output_dir, 'all_alpha_carbons_02.pkl')
with open(output_path_02, 'wb') as f:
    pickle.dump(all_alpha_carbons_02, f)
print(f"Saved all_alpha_carbons_02 to {output_path_02}")

Saved all_alpha_carbons_01 to data_processed/all_alpha_carbons_01.pkl
Saved all_alpha_carbons_02 to data_processed/all_alpha_carbons_02.pkl


In [ ]:
graphs_01 = []
print("Generating graphs for oxireductases.")
for pdb_id, ca_coords in tqdm(all_alpha_carbons_01):
  if ca_coords:
    edge_index = generate_graph(ca_coords)
    graphs_01.append((pdb_id, edge_index))
  else:
    print(f"Warning: No alpha carbons found for {pdb_id} in EC_01. Skipping graph generation.")

Generating graphs for oxireductases.


100%|██████████| 471/471 [19:43<00:00,  2.51s/it]


In [ ]:
graphs_02 = []
print("Generating graphs for transferases.")
for pdb_id, ca_coords in tqdm(all_alpha_carbons_02):
  if ca_coords:
    edge_index = generate_graph(ca_coords)
    graphs_02.append((pdb_id, edge_index))
  else:
    print(f"Warning: No alpha carbons found for {pdb_id} in EC_02. Skipping graph generation.")

Generating graphs for transferases.


100%|██████████| 752/752 [19:03<00:00,  1.52s/it]


In [ ]:
output_path_graphs_01 = os.path.join(output_dir, 'graphs_01.pkl')
with open(output_path_graphs_01, 'wb') as f:
  pickle.dump(graphs_01, f)
print(f"Saved graphs_01 to {output_path_graphs_01}")

output_path_graphs_02 = os.path.join(output_dir, 'graphs_02.pkl')
with open(output_path_graphs_02, 'wb') as f:
  pickle.dump(graphs_02, f)
print(f"Saved graphs_02 to {output_path_graphs_02}")

Saved graphs_01 to data_processed/graphs_01.pkl
Saved graphs_02 to data_processed/graphs_02.pkl


In [5]:
import pickle

all_alpha_carbons_01 = pickle.load(open('data_processed/all_alpha_carbons_01.pkl', 'rb'))
all_alpha_carbons_02 = pickle.load(open('data_processed/all_alpha_carbons_02.pkl', 'rb'))
graphs_01 = pickle.load(open('data_processed/graphs_01.pkl', 'rb'))
graphs_02 = pickle.load(open('data_processed/graphs_02.pkl', 'rb'))

## 2. Data Manipulation - Making the data ready for Machine Learning

We need everything ready for Pytorch.

Given a graph $G$ with $n$ vertices, each vertex is represented by a number from $0$ to $n-1$.

Each vertex is associated with a feature vector that contains information about this vertex, let $m$ be the dimension of these vectors. The complete set of features will be represented by a Pytorch tensor $H$ of dimensions $n \times m$, where each line $H_{i,:}$ is the vector of size $m$ associated with vertex $i$.

The edges of the graph will be represented by an edge list, a tensor of size $e \times 2$ where $e$ is the number of edges in the graph, when a column has entries $i$ and $j$ it means that these vertices are connected in the graph.



### 2.1 Encondig vertex features

Proteins are composed of amino acids linked by peptide bonds, each vertex of our graph corresponds from an alpha-carbon from an amino acid. The initial feature vector of each vertex will encode the type of the corresponding amino acid. We will use one-hot encoding in 21 classes: one for each canonical amino acid and one for non-canonical amino-acids.

The function below takes as input a list of $n$ amino acids of a protein, and outputs a $n \times 21$ matrix that is the matrix of the features of the vertices of the associated graph.

In [6]:
import torch
import torch.nn.functional as F

def one_hot_encoding(aa_list):
  canonical_aa = [
          'GLY', 'ALA', 'VAL', 'LEU', 'ILE', 'PRO', 'MET',
          'PHE', 'TRP', 'TYR', 'SER', 'THR', 'CYS', 'ASN',
          'GLN', 'LYS', 'ARG', 'HIS', 'ASP', 'GLU'
      ]

  aa_to_idx = {aa:i for i, aa in enumerate(canonical_aa)}

  indices = torch.tensor([aa_to_idx.get(aa.upper(), 20) for aa in aa_list], dtype=torch.long)

  one_hot = F.one_hot(indices, num_classes=21)

  return one_hot

Since we have 471 hydrolase examples and 752 transferase examples, we will undersample to obtain balanced classes. We will select 350, 50 and 50 examples from each class for the training, validation and testing datasets.

In [7]:
import random

n_train, n_val, n_test = 350, 50, 50

def get_split_idx(total, n_train, n_val, n_test):
  indices = list(range(total))
  random.shuffle(indices)

  train_idx = indices[:n_train]
  val_idx = indices[n_train:n_train+n_val]
  test_idx = indices[n_train+n_val:n_train+n_val+n_test]

  return train_idx, val_idx, test_idx

Now we will create the datasets. PyTorch Geometric has the class Data for storing graphs, each dataset will be a list of Data objects.

In [8]:
from torch_geometric.data import Data

hydrolase_idx = get_split_idx(len(all_alpha_carbons_01), n_train, n_val, n_test)
transferase_idx = get_split_idx(len(all_alpha_carbons_02), n_train, n_val, n_test)

testing_dataset = []
validation_dataset = []
training_dataset = []

for i in range(len(all_alpha_carbons_01)):

  assert all_alpha_carbons_01[i][0] == graphs_01[i][0]
  graph = Data(
      x = one_hot_encoding([all_alpha_carbons_01[i][1][_][0] for _ in range(len(all_alpha_carbons_01[i][1]))]),
      edge_index = torch.from_numpy(graphs_01[i][1]),
      y=0
  )

  if i in hydrolase_idx[0]:
    training_dataset.append(graph)
  elif i in hydrolase_idx[1]:
    validation_dataset.append(graph)
  elif i in hydrolase_idx[2]:
    testing_dataset.append(graph)

for i in range(len(all_alpha_carbons_02)):

  assert all_alpha_carbons_02[i][0] == graphs_02[i][0]
  graph = Data(
      x = one_hot_encoding([all_alpha_carbons_02[i][1][_][0] for _ in range(len(all_alpha_carbons_02[i][1]))]),
      edge_index = torch.from_numpy(graphs_02[i][1]),
      y=1
  )

  if i in transferase_idx[0]:
    training_dataset.append(graph)
  elif i in transferase_idx[1]:
    validation_dataset.append(graph)
  elif i in transferase_idx[2]:
    testing_dataset.append(graph)

random.shuffle(training_dataset)
random.shuffle(validation_dataset)
random.shuffle(testing_dataset)

Having the datasets, we can create the data loaders.

In [9]:
from torch_geometric.loader import DataLoader

batch_size = 32

training_loader = DataLoader(training_dataset, batch_size=batch_size, shuffle=True)
validation_loader   = DataLoader(validation_dataset, batch_size=batch_size, shuffle=False)
testing_loader  = DataLoader(testing_dataset, batch_size=batch_size, shuffle=False)

## 3. Model Architecture

### 3.1 Graph Convolution Layers

We will construct little by little what will the be a graph convolution layer.

Let $H$ be the $n \times m$ matrix of the features vectors, the $i$-th row of $H$, the vector $H_{i,:}$, is the feature vector of the vertex $i$.

A simple update for the feature vectors would be the following. Given a feature vector $H_{i,:}$ we can update it by the sum of itself with the feature vectors of the neighbors of $i$. That is,

$$H_{i,:} := \sum_{i,j \text{ are neighbors}} H_{j,:}$$

where we are considering that each vertex is neighbor to itself.

Having learnable parameters is what makes machine learning, machine learning. So we introduce a learnable matrix $W$, with dimensions $n \times n$. These dimensions will preserve the dimension of $H$.

$$H_{i,:} := \left(\sum_{i,j \text{ are neighbors}} H_{j,:}\right)W$$

Learnable parameters is the essence of machine learning, non-linearity is what allows it to be powerfully expressible. Thus, we introduce non-linearity through an activation function:

$$H_{i,:} := \sigma\left(\left(\sum_{i,j \text{ are neighbors}} H_{j,:}\right)W\right)$$

Now, the final touch. Since each feature vector is being update by the sum of itself with its neighbors, they can become very large after many layers. So we introduce the scaling factors:

$$H_{i,:} = \sigma\left(\left(\left(\frac{1}{\sqrt{d_i}}\sum_{i,j \text{ are neighbors}} \frac{1}{\sqrt{d_j}}H_{j,:}\right)W\right)\right)$$

where $d_i$ is the degree of the vertex $i$.

Why $\frac{1}{\sqrt{d_i}}$ and $\frac{1}{\sqrt{d_j}}$? Lets save that for another ocasion.

As an exercise, you can show that this is equivalent to

$$H := \sigma(D^{-1/2}HD^{-1/2}W)$$

where $D$ is the diagonal matrix where $D_{ii} = d_i$.

This approach is implemented in the class GCNConv of PyTorch Geometric, following the paper  [Semi-supervised Classification with Graph Convolutional Networks](https://https://arxiv.org/abs/1609.02907).

###3.2 Implement model architecture

Our model architecture will be simple. We stack three graph convolution layers, take an average of all the resulting feature vectors, apply a linear transformation with a single dimension as output.

Observe that since we have only 3 graph convolution layers, then the vertices only can exchange information with vertices that have distance 3 or less to them.

In [10]:
import torch.nn as nn
from torch_geometric.nn import GCNConv, global_mean_pool

class EnzymeGNN(nn.Module):
  def __init__(self):
    super(EnzymeGNN, self).__init__()

    self.conv1 = GCNConv(21, 21)
    self.conv2 = GCNConv(21, 21)
    self.conv3 = GCNConv(21, 21)
    self.conv4 = GCNConv(21, 21)
    self.conv5 = GCNConv(21, 21)

    self.linear = nn.Linear(21, 1)

  def forward(self, x, edge_index, batch):
    x = self.conv1(x, edge_index)
    x = x.relu()
    x = self.conv2(x, edge_index)
    x = x.relu()
    x = self.conv3(x, edge_index)
    x = x.relu()
    x = self.conv4(x, edge_index)
    x = x.relu()
    x = self.conv5(x, edge_index)

    x = global_mean_pool(x, batch)
    x = self.linear(x)

    return x

## 4. Model Training

In [11]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = EnzymeGNN()
model = model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = torch.nn.BCEWithLogitsLoss()

In [12]:
def train():
    model.train()
    total_loss = 0
    for data in training_loader:
        data = data.to(device)
        optimizer.zero_grad()

        # Forward pass
        # The model returns logits of shape [batch_size, 1]
        out = model(data.x.float(), data.edge_index, data.batch)

        # Compute loss (reshaping y to match out shape)
        loss = criterion(out, data.y.float().view(-1, 1))

        # Backward pass
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * data.num_graphs

    return total_loss / len(training_loader.dataset)

@torch.no_grad()
def test(loader):
    model.eval()
    correct = 0
    for data in loader:
        data = data.to(device)
        out = model(data.x.float(), data.edge_index, data.batch)

        # Apply sigmoid or simply check if logit > 0 for binary classification
        pred = (out > 0).float()
        correct += (pred == data.y.float().view(-1, 1)).sum().item()

    return correct / len(loader.dataset)

# Main Training Loop
num_epochs = 100
print("Starting training...")

for epoch in range(1, num_epochs + 1):
    loss = train()
    train_acc = test(training_loader)
    val_acc = test(validation_loader)

    if epoch % 1 == 0 or epoch == 1:
        print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}')

# Final evaluation on the test set
test_acc = test(testing_loader)
print(f'\nFinal Test Accuracy: {test_acc:.4f}')

Starting training...
Epoch: 001, Loss: 0.6938, Train Acc: 0.5000, Val Acc: 0.5000
Epoch: 002, Loss: 0.6935, Train Acc: 0.5000, Val Acc: 0.5000
Epoch: 003, Loss: 0.6927, Train Acc: 0.5657, Val Acc: 0.5000
Epoch: 004, Loss: 0.6923, Train Acc: 0.5000, Val Acc: 0.5000
Epoch: 005, Loss: 0.6904, Train Acc: 0.6886, Val Acc: 0.7000
Epoch: 006, Loss: 0.6883, Train Acc: 0.6457, Val Acc: 0.6400
Epoch: 007, Loss: 0.6826, Train Acc: 0.6057, Val Acc: 0.6000
Epoch: 008, Loss: 0.6748, Train Acc: 0.6043, Val Acc: 0.6100
Epoch: 009, Loss: 0.6619, Train Acc: 0.6871, Val Acc: 0.6400
Epoch: 010, Loss: 0.6407, Train Acc: 0.6143, Val Acc: 0.6100
Epoch: 011, Loss: 0.6347, Train Acc: 0.6843, Val Acc: 0.6400
Epoch: 012, Loss: 0.6132, Train Acc: 0.7086, Val Acc: 0.7100
Epoch: 013, Loss: 0.6077, Train Acc: 0.6714, Val Acc: 0.5700
Epoch: 014, Loss: 0.6066, Train Acc: 0.7029, Val Acc: 0.7000
Epoch: 015, Loss: 0.5808, Train Acc: 0.6886, Val Acc: 0.6100
Epoch: 016, Loss: 0.5777, Train Acc: 0.6900, Val Acc: 0.6800
Epo